### Problem Statement


The data is related to stocks listed on a stock market. There are 103 unique stocks in the provided dataset which contains stocks that are listed on a public stock exchange, as well as those that are traded privately, such as shares of private companies which are sold to investors through equity crowdfunding platforms.

The documents in the dataset have the following fields -

- stock - Unique ID for a stock

- date - Date for which the stock prices are provided

- open - Opening price of the stock on a trading day

- high - Highest price of the stock on a trading day

- low - Lowest price of the stock on a trading day

- close - Closing price of the stock on a trading day

- holiday - Whether there is a holiday or not (0/1)

----

### Connecting to MongoDB


----

In [1]:
# Importing the required libraries
import pymongo
import pprint as pp

pp.sorted = lambda x, key=None: x

In [2]:
client = pymongo.MongoClient("mongodb://localhost:27017/")

---
### Importing data

----

In [3]:
# # Restore database
# !mongorestore /home/avadmin/Desktop/Mongo/Content/Aggregation/Assignment/Data/aggregation_assignment

In [4]:
# Database
db = client.Project

In [5]:
# List collections
db.list_collection_names()

['stock', 'Assignment']

In [6]:
# Sample document
pp.pprint(
    db.stock.find_one()
)

{'_id': '60d9930f5922a69299cb36c7',
 'stock': 0,
 'date': '2017-01-03T00:00:00',
 'open': 82.9961,
 'high': 82.7396,
 'low': 82.9144,
 'close': 82.8101,
 'holiday': 1}


----
### Assignment Questions

----

###  Q1

Average `open`, `close`, `high` and `low` price of each stock

In [7]:
pipeline_1 = [
    { "$group": { "_id": "$stock",
            "avg_open": {"$avg": "$open"},
            "avg_close": {"$avg": "$close"},
            "avg_high": {"$avg": "$high"},
            "avg_low": {"$avg": "$low"} } },
    {
        "$sort": {"_id": 1}  # sort (stock ID) 
    }   ]

result_1 = list(db.stock.aggregate(pipeline_1))

for stock in result_1:
    print(f"Stock ID : {stock['_id']}")
    print(f"Avg Open : {stock['avg_open']:.2f}")
    print(f"Avg Close : {stock['avg_close']:.2f}")
    print(f"Avg High : {stock['avg_high']:.2f}")
    print(f"Avg Low : {stock['avg_low']:.2f}")
    print("-" * 30)

Stock ID : 0
Avg Open : 95.89
Avg Close : 95.95
Avg High : 96.06
Avg Low : 95.83
------------------------------
Stock ID : 1
Avg Open : 41.56
Avg Close : 41.60
Avg High : 41.41
Avg Low : 41.81
------------------------------
Stock ID : 2
Avg Open : 140.34
Avg Close : 140.43
Avg High : 139.85
Avg Low : 141.07
------------------------------
Stock ID : 3
Avg Open : 68.95
Avg Close : 69.06
Avg High : 68.85
Avg Low : 69.24
------------------------------
Stock ID : 4
Avg Open : 77.82
Avg Close : 77.81
Avg High : 78.15
Avg Low : 77.55
------------------------------
Stock ID : 5
Avg Open : 120.21
Avg Close : 120.27
Avg High : 120.95
Avg Low : 119.48
------------------------------
Stock ID : 6
Avg Open : 112.05
Avg Close : 112.22
Avg High : 111.75
Avg Low : 112.67
------------------------------
Stock ID : 7
Avg Open : 143.21
Avg Close : 143.36
Avg High : 142.90
Avg Low : 143.84
------------------------------
Stock ID : 8
Avg Open : 59.03
Avg Close : 59.10
Avg High : 58.96
Avg Low : 59.24
-------

### Q2

Top 10 stocks that have the highest maximum `high` price.

In [8]:
pipeline = [
    { "$group": { "_id": "$stock", "max_high": {"$max": "$high"} } },
    { "$sort": {"max_high": -1 }  # sort with descending order 
    },
    { "$limit": 10  # only top 10 results
    }  ]

result = list(db.stock.aggregate(pipeline))

for stock in result:
    print(f"Stock ID : {stock['_id']}")
    print(f"Max High Price : {stock['max_high']:.2f}")
    print("-" * 30)

Stock ID : 50
Max High Price : 233.65
------------------------------
Stock ID : 9
Max High Price : 188.87
------------------------------
Stock ID : 96
Max High Price : 172.62
------------------------------
Stock ID : 41
Max High Price : 169.87
------------------------------
Stock ID : 5
Max High Price : 163.39
------------------------------
Stock ID : 98
Max High Price : 162.53
------------------------------
Stock ID : 83
Max High Price : 157.79
------------------------------
Stock ID : 7
Max High Price : 155.51
------------------------------
Stock ID : 101
Max High Price : 152.08
------------------------------
Stock ID : 66
Max High Price : 151.10
------------------------------


### Q3

Find the number of days for each stock for which it had `close` price more than the `open` price.

In [9]:
pipeline_2 = [
    { "$match": { "$expr": { "$gt": ["$close", "$open"] } } },
    { "$group": { "_id": "$stock", "days_close_gt_open": { "$sum": 1 } } },
    {
        "$sort": {"_id": 1}  # Sort in order of stock ID
    }  ]

result_2 = list(db.stock.aggregate(pipeline_2))

for stock in result_2:
    print(f"Stock ID : {stock['_id']}")
    print(f"Days Close > Open : {stock['days_close_gt_open']}")
    print("-" * 30)

Stock ID : 0
Days Close > Open : 409
------------------------------
Stock ID : 1
Days Close > Open : 601
------------------------------
Stock ID : 2
Days Close > Open : 510
------------------------------
Stock ID : 3
Days Close > Open : 499
------------------------------
Stock ID : 4
Days Close > Open : 366
------------------------------
Stock ID : 5
Days Close > Open : 400
------------------------------
Stock ID : 6
Days Close > Open : 516
------------------------------
Stock ID : 7
Days Close > Open : 475
------------------------------
Stock ID : 8
Days Close > Open : 476
------------------------------
Stock ID : 9
Days Close > Open : 384
------------------------------
Stock ID : 10
Days Close > Open : 449
------------------------------
Stock ID : 11
Days Close > Open : 534
------------------------------
Stock ID : 12
Days Close > Open : 400
------------------------------
Stock ID : 13
Days Close > Open : 461
------------------------------
Stock ID : 14
Days Close > Open : 629
------

### Q4

Find average `open` price, using all the stocks, for each of the following -

- for each year. Ex - for 2017, 2018, etc.

![](Images/shot_1.png)

- for every month of each year. Ex - for January 2017, February 2017, etc.

![](Images/shot_2.png)

- for every week of each year. Ex - for Week 1 2017, Week 2 2017, etc.

![](Images/shot_3.png)

***Hint - Use a single pipeline to answer the above query.***


In [10]:
pipeline_3 = [
    
 { "$addFields": { "date_obj": { "$dateFromString": { "dateString": "$date" } } } },
    
     { "$facet": 
            { 
                
        "yearly_avg": [ 
                            { "$group": { "_id": { "$year": "$date_obj" }, "avg_open": { "$avg": "$open" } } },
                            { "$sort": { "_id": 1 } }  ],
                 
        "monthly_avg": [
                            { "$group": { "_id": { "year": { "$year": "$date_obj" }, "month": { "$month": "$date_obj" } },
                            "avg_open": { "$avg": "$open" }  } },
                            { "$sort": { "_id.year": 1, "_id.month": 1 } }  ],
                 
        "weekly_avg": [
                            { "$group": { "_id": { "year": { "$isoWeekYear": "$date_obj" }, "week": { "$isoWeek": "$date_obj" } }, "avg_open": { "$avg": "$open" } } },
                { "$sort": { "_id.year": 1, "_id.week": 1 } }  ]
        }
    }
]

final_result = list(db.stock.aggregate(pipeline_3))[0]

# Print yearly average
print("Yearly Average Open Prices:")
for item in final_result["yearly_avg"]:
    print(f"Year {item['_id']} : {item['avg_open']:.2f}")

print("\nMonthly Average Open Prices:")
for item in final_result["monthly_avg"]:
    print(f"{item['_id']['month']:02d}-{item['_id']['year']} : {item['avg_open']:.2f}")

print("\nWeekly Average Open Prices:")
for item in final_result["weekly_avg"]:
    print(f"Week {item['_id']['week']:02d}-{item['_id']['year']} : {item['avg_open']:.2f}")


Yearly Average Open Prices:
Year 2017 : 70.74
Year 2018 : 77.16
Year 2019 : 78.88

Monthly Average Open Prices:
01-2017 : 65.56
02-2017 : 66.85
03-2017 : 67.40
04-2017 : 67.90
05-2017 : 69.40
06-2017 : 70.34
07-2017 : 71.29
08-2017 : 71.46
09-2017 : 72.74
10-2017 : 74.18
11-2017 : 74.90
12-2017 : 76.58
01-2018 : 80.09
02-2018 : 77.41
03-2018 : 77.49
04-2018 : 77.05
05-2018 : 77.47
06-2018 : 77.91
07-2018 : 77.82
08-2018 : 78.67
09-2018 : 79.06
10-2018 : 75.90
11-2018 : 74.73
12-2018 : 71.98
01-2019 : 73.34
02-2019 : 76.92
03-2019 : 78.13
04-2019 : 80.22
05-2019 : 78.04
06-2019 : 79.01
07-2019 : 81.44
08-2019 : 79.01
09-2019 : 81.19
10-2019 : 81.10

Weekly Average Open Prices:
Week 01-2017 : 64.97
Week 02-2017 : 65.24
Week 03-2017 : 65.49
Week 04-2017 : 66.17
Week 05-2017 : 66.28
Week 06-2017 : 66.31
Week 07-2017 : 67.05
Week 08-2017 : 67.41
Week 09-2017 : 67.47
Week 10-2017 : 67.10
Week 11-2017 : 67.44
Week 12-2017 : 67.48
Week 13-2017 : 67.48
Week 14-2017 : 67.68
Week 15-2017 : 67.53


### Q5

Find the number of stocks that have `high` price in the groups [0, 50), [50, 100), [100, 200) and [200, 400).

***Note - [ means value is included. And ) means that the value is excluded.***

In [11]:
pipline_high_price = [
    {
        "$project": {
            "stock": 1,
            "high": 1,
            "range": {
                "$switch": {
                    "branches": [
                        {
                            "case": { "$and": [ { "$gte": ["$high", 0] }, { "$lt": ["$high", 50] } ] },
                            "then": "[0, 50)"
                        },
                        {
                            "case": { "$and": [ { "$gte": ["$high", 50] }, { "$lt": ["$high", 100] } ] },
                            "then": "[50, 100)"
                        },
                        {
                            "case": { "$and": [ { "$gte": ["$high", 100] }, { "$lt": ["$high", 200] } ] },
                            "then": "[100, 200)"
                        },
                        {
                            "case": { "$and": [ { "$gte": ["$high", 200] }, { "$lt": ["$high", 400] } ] },
                            "then": "[200, 400)"
                        }
                    ],
                    "default": "other"
                }
            }
        }
    },
    {
        "$match": { "range": { "$ne": "other" } }
    },
    {
        "$group": {
            "_id": { "range": "$range", "stock": "$stock" }
        }
    },
    {
        "$group": {
            "_id": "$_id.range",
            "unique_stocks": { "$sum": 1 }
        }
    },
    {
        "$sort": { "_id": 1 }
    }
]

result_4 = list(db.stock.aggregate(pipline_high_price))

for item in result_4:
    print(f"Range {item['_id']} : {item['unique_stocks']} unique stocks")

Range [0, 50) : 42 unique stocks
Range [100, 200) : 37 unique stocks
Range [200, 400) : 1 unique stocks
Range [50, 100) : 82 unique stocks


### Q6

Count the number of holidays.

In [12]:
check_holidays = [
    {
        "$match": { "holiday": 1 }
    },
    {
        "$count": "total_holidays"
    }
]

final = list(db.stock.aggregate(check_holidays))

if final:
    print(f"Total Holidays : {final[0]['total_holidays']}")
else:
    print("No holidays found.")

Total Holidays : 4223
